In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcryff.trainer import ThermodynamicTrainer
from imolcryff.trainer.dmff_utils import parser_dmffyaml
from imolcryff.trainer.loss import loss_thermodynamicperturbation
from functools import partial

/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/mdtraj/geometry/order.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/srak/apl/iMolCRAFT/DMFF/dmff/admp/qeq.py:33: UserWarning: jaxopt not found, QEQ cannot be used.
  warnings.warn("jaxopt not found, QEQ cannot be used.")
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 1.2,
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 30,
  'anneal_totaltime': 0},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 36.82},
  'Lc_A': {'weight': 1.0, 'gt': 40.2152},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="merged_supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charges"],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [6]:
from imolcryff.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[10].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[11].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[12].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[13].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[14].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[15].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[16].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[17].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[18].set(0.0)
    grads["NonbondedForce"]["charges"] = grads["NonbondedForce"]["charges"].at[19].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9]], target_charges=[0.0])

    return ffparams

In [7]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [8]:
trainer.setup()

Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5014579474321446,0,--
2000,1.5047621695458173,185,0:55
3000,1.496217184310231,183,0:55
4000,1.4967178548640585,182,0:55
5000,1.5021407414483787,182,0:54
6000,1.4968087354177273,182,0:54
7000,1.4967903543716787,181,0:53
8000,1.4952364255593837,181,0:53
9000,1.502928009435361,181,0:52
10000,1.494486759221586,181,0:52
11000,1.4940209456939226,182,0:51
12000,1.4942696467503669,182,0:51
13000,1.490888056357567,182,0:50
14000,1.492566018937195,182,0:50
15000,1.4950577362241013,182,0:49
16000,1.4982434094036408,182,0:49
17000,1.4889721335751007,183,0:48
18000,1.4957263192729484,182,0:48
19000,1.4919938439353502,182,0:47
20000,1.4934687861714155,182,0:47
== Start Production ==
21000,1.4931895402187736,182,0:46
22000,1.4923191767311472,181,0:46
23000,1.4875574536841751,180,0:46
24000,1.4959233342048153,180,0:45
25000,1.4909569080

100%|██████████| 100/100 [00:02<00:00, 38.39it/s]


In [11]:
neff_prev = trainer.neff
print("Initial neff:", neff_prev)

Initial neff: [30]


In [12]:
trainer.neff = [50]

In [13]:
trainer.fit(10, 2)

100%|██████████| 100/100 [00:19<00:00,  5.13it/s]


Effective sample sizes:
  sample_0: 59
  Total: 59.01506836116177
Epoch 0 completed in 100.83 seconds.
Loss:  1.0468162206500842
Best Loss:  1.0468162206500842 at epoch  0


100%|██████████| 100/100 [00:18<00:00,  5.35it/s]


Effective sample sizes:
  sample_0: 59
  Total: 59.730940684084025
Epoch 1 completed in 65.58 seconds.
Loss:  1.0465791325135996
Best Loss:  1.0465791325135996 at epoch  1


100%|██████████| 100/100 [00:19<00:00,  5.02it/s]


...

Effective sample sizes:
  sample_0: 63
  Total: 63.20235763202892
Epoch 8 completed in 95.15 seconds.
Loss:  1.045020606868722
Best Loss:  1.045020606868722 at epoch  8


100%|██████████| 100/100 [00:24<00:00,  4.01it/s]


Effective sample sizes:
  sample_0: 63
  Total: 63.429402188908405
Epoch 9 completed in 80.69 seconds.
Loss:  1.0448219352486245
Best Loss:  1.0448219352486245 at epoch  9


100%|██████████| 100/100 [00:34<00:00,  2.91it/s]


Effective sample sizes:
  sample_0: 63
  Total: 63.5806983425966
Epoch 10 completed in 99.80 seconds.
Loss:  1.0446302645548364
Best Loss:  1.0446302645548364 at epoch  10


In [ ]:
trainer.neff = neff_prev

In [ ]:
trainer.fit(490, 2)

100%|██████████| 100/100 [00:19<00:00,  5.05it/s]


Effective sample sizes:
  sample_0: 58
  Total: 59.0404343234578
Epoch 5 completed in 70.17 seconds.
Loss:  1.0353482554177909
Best Loss:  1.0353482554177909 at epoch  5


100%|██████████| 100/100 [00:19<00:00,  5.06it/s]


Effective sample sizes:
  sample_0: 57
  Total: 58.36476931370636
Epoch 6 completed in 70.69 seconds.
Loss:  1.0333002841590788
Best Loss:  1.0333002841590788 at epoch  6


100%|██████████| 100/100 [00:19<00:00,  5.10it/s]


...

Effective sample sizes:
  sample_0: 45
  Total: 46.024393372112094
Epoch 43 completed in 72.12 seconds.
Loss:  1.0129508009412858
Best Loss:  1.0129508009412858 at epoch  43


100%|██████████| 100/100 [00:18<00:00,  5.28it/s]


Effective sample sizes:
  sample_0: 43
  Total: 44.202465179610165
Epoch 44 completed in 73.42 seconds.
Loss:  1.0118735525668716
Best Loss:  1.0118735525668716 at epoch  44


100%|██████████| 100/100 [00:19<00:00,  5.19it/s]


Effective sample sizes:
  sample_0: 41
  Total: 42.36731841777646
Epoch 45 completed in 72.30 seconds.
Loss:  1.010798988817868
Best Loss:  1.010798988817868 at epoch  45


In [11]:
trainer.fit(40, 2)

100%|██████████| 100/100 [00:19<00:00,  5.11it/s]


Effective sample sizes:
  sample_0: 39
  Total: 40.54647117330796
Epoch 46 completed in 72.60 seconds.
Loss:  1.0097279373282642
Best Loss:  1.0097279373282642 at epoch  46


100%|██████████| 100/100 [00:19<00:00,  5.12it/s]


Effective sample sizes:
  sample_0: 37
  Total: 38.765157724171694
Epoch 47 completed in 72.48 seconds.
Loss:  1.0086608095295626
Best Loss:  1.0086608095295626 at epoch  47


100%|██████████| 100/100 [00:19<00:00,  5.12it/s]


...

Effective sample sizes:
  sample_0: 43
  Total: 44.62604966067916
Epoch 84 completed in 75.53 seconds.
Loss:  1.0091888654318382
Best Loss:  0.9818966544421401 at epoch  72


100%|██████████| 100/100 [00:20<00:00,  4.89it/s]


Effective sample sizes:
  sample_0: 43
  Total: 44.2956526486657
Epoch 85 completed in 74.43 seconds.
Loss:  1.0086124103100402
Best Loss:  0.9818966544421401 at epoch  72


100%|██████████| 100/100 [00:20<00:00,  4.81it/s]


Effective sample sizes:
  sample_0: 42
  Total: 43.99491679284717
Epoch 86 completed in 75.31 seconds.
Loss:  1.0080340787767201
Best Loss:  0.9818966544421401 at epoch  72


In [ ]:
# trainer.from_checkpoint(trainer_checkpoint="train_state.pkl",
#                         ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
#                         nums_ffxml=[2,1,1],
#                         pdbfile="merged_supercell_bonds.pdb",
#                         initial_ffxml="xmlfiles/epoch-4.xml",
#                         loss_fn=lossfn,
#                         opt_fftypes=["NonbondedForce/charges"],
#                         )